# 声速测量计算器 (逐差法)

本 Notebook 用于通过**干涉法（驻波法）**与**相位法（李萨如图形法）**测得的接收器位移坐标，利用**逐差法**计算超声波在空气中的波长 $\lambda$、实际声速 $v$，并与理论声速进行对比和不确定度评定。

---
### 实验原理与数学公式

#### 1. 理论声速公式
在摄氏温度 $t\,({}^\circ\mathrm{C})$ 下，空气中理想声速为：
$$v_t = 331.45 \sqrt{1 + \frac{t}{273.15}} \quad (\mathrm{m/s})$$

#### 2. 逐差法测波长 $\lambda$
两组数据各测 12 个极值或特定相位位置坐标 $x_1, x_2, \dots, x_{12}$，分成两组逐差：
$$\Delta x_i = x_{i+6} - x_i \quad (i = 1, 2, \dots, 6), \quad \Delta x_{\mathrm{avg}} = \frac{1}{6} \sum_{i=1}^6 \Delta x_i$$
* **干涉法（驻波法）**：相邻波腹间距为 $\lambda/2$，6 个间隔对应 $\Delta x = 6 \times (\lambda/2) = 3\lambda$，故：
$$\lambda = \frac{\Delta x_{\mathrm{avg}}}{3}$$
* **相位法（李萨如法）**：相邻同向直线相位差为 $2\pi$，对应间距为 $\lambda$，6 个间隔对应 $\Delta x = 6\lambda$，故：
$$\lambda = \frac{\Delta x_{\mathrm{avg}}}{6}$$

#### 3. 声速与不确定度传递公式
实验测得声速为 $v = f \cdot \lambda$。
* 逐差平均值的 A 类不确定度：$u_A(\Delta x_{\mathrm{avg}}) = \frac{s_\Delta}{\sqrt{6}}$
* 逐差平均值的 B 类不确定度：$u_B(\Delta x_{\mathrm{avg}}) = \sqrt{\frac{2}{6}} \frac{\Delta_{\mathrm{inst}}}{\sqrt{3}} = \frac{\Delta_{\mathrm{inst}}}{3}$
* 合成不确定度：$u_c(\Delta x_{\mathrm{avg}}) = \sqrt{u_A^2 + u_B^2}$
* 波长及声速不确定度（忽略频率 $f$ 极小的不确定度）：
$$u(\lambda) = \frac{u_c(\Delta x_{\mathrm{avg}})}{k}, \quad u(v) = f \cdot u(\lambda)$$

In [ ]:
import math
from decimal import Decimal
from python.utils import scientific_round

print("声速计算模块加载完成。")

### 1. 实验条件与测量数据输入
> **提示**：可在此输入环境温度、激励信号源频率，以及干涉法和相位法分别测得的 12 个坐标值 $x_i\,(\mathrm{mm})$。

In [ ]:
# 实验环境参数
t_celsius = 24.0             # 环境温度 (℃)
freq = 37000.0               # 谐振超声波频率 (Hz)
delta_inst = Decimal("0.02") # 游标卡尺/测微鼓轮分度值误差 (mm)

# 1. 干涉法 (驻波法) 12 个坐标 (mm)
x_vals_interference = [
    Decimal("10.20"), Decimal("14.85"), Decimal("19.50"), Decimal("24.15"), Decimal("28.80"), Decimal("33.45"),
    Decimal("38.10"), Decimal("42.75"), Decimal("47.40"), Decimal("52.05"), Decimal("56.70"), Decimal("61.35")
]

# 2. 相位法 (李萨如图形法) 12 个坐标 (mm)
x_vals_phase = [
    Decimal("12.00"), Decimal("21.30"), Decimal("30.60"), Decimal("39.90"), Decimal("49.20"), Decimal("58.50"),
    Decimal("67.80"), Decimal("77.10"), Decimal("86.40"), Decimal("95.70"), Decimal("105.00"), Decimal("114.30")
]

v_theory = 331.45 * math.sqrt(1 + t_celsius / 273.15)
print(f"当前温度 {t_celsius} ℃ 下的理论声速 v_t: {v_theory:.2f} m/s")

### 2. 逐差法处理与声速不确定度评定

In [ ]:
def process_speed_of_sound(method_name, x_vals, f, k, delta_inst):
    M = 6
    deltas = []
    for i in range(M):
        diff = x_vals[i + M] - x_vals[i]
        deltas.append(diff)
    mean_delta = sum(deltas) / Decimal(str(M))
    
    # 计算波长 (mm -> m)
    wavelength_mm = mean_delta / Decimal(str(k))
    wavelength_m = wavelength_mm / Decimal("1000")
    velocity = Decimal(str(f)) * wavelength_m
    
    # 不确定度分析
    variance_delta = sum((d - mean_delta)**2 for d in deltas) / Decimal(str(M - 1))
    s_delta = Decimal(str(math.sqrt(float(variance_delta))))
    u_a_delta = s_delta / Decimal(str(math.sqrt(M)))
    
    u_x = delta_inst / Decimal(str(math.sqrt(3)))
    u_b_delta = Decimal(str(math.sqrt(2.0 / M))) * u_x
    u_delta_combined = Decimal(str(math.sqrt(float(u_a_delta**2 + u_b_delta**2))))
    
    u_wavelength_mm = u_delta_combined / Decimal(str(k))
    u_wavelength_m = u_wavelength_mm / Decimal("1000")
    u_velocity = Decimal(str(f)) * u_wavelength_m
    
    v_final, uv_final = scientific_round(velocity, u_velocity)
    
    print("=" * 45)
    print(f"         {method_name} 计算结果         ")
    print("=" * 45)
    print(f"逐差平均位移 Δx_avg : {mean_delta:.4f} mm")
    print(f"位移标准差 s(Δx)    : {s_delta:.4f} mm")
    print(f"声波波长 λ          : {wavelength_mm:.4f} mm")
    print(f"波长不确定度 u(λ)   : {u_wavelength_mm:.4f} mm")
    print(f"实验声速 v (原始)   : {velocity:.2f} m/s")
    print(f"声速合成不确定度 u(v): {u_velocity:.2f} m/s")
    rel_error = abs(float(velocity) - v_theory) / v_theory * 100
    print(f"与理论声速相对误差  : {rel_error:.2f}%")
    print("-" * 45)
    print(f"★ 最终修约结果      : v = {v_final} ± {uv_final} m/s")
    print("=" * 45 + "\n")

# 处理两种方法
process_speed_of_sound("干涉法 (驻波法)", x_vals_interference, freq, 3, delta_inst)
process_speed_of_sound("相位法 (李萨如法)", x_vals_phase, freq, 6, delta_inst)